In [0]:
%sql
-- Cria tabela Silver de clientes com schema tipado (vazia)
CREATE TABLE IF NOT EXISTS `desafio-semana-2`.silver.silver_clientes (
    id_cliente INT,
    nome STRING,
    sexo STRING,
    data_nascimento DATE,
    cidade STRING,
    estado STRING,
    data_cadastro DATE,
    dh_atualizacao TIMESTAMP
) USING DELTA;

In [0]:
%sql
-- MERGE: tipagem (INT, DATE), padronização (TRIM, INITCAP, UPPER), tratamento de nulos
-- BONUS: idempotente — pode reexecutar sem duplicar dados
MERGE INTO `desafio-semana-2`.silver.silver_clientes AS dest
USING (
    SELECT
        try_cast(id_cliente AS INT) AS id_cliente,
        TRIM(nome) AS nome,
        INITCAP(TRIM(sexo)) AS sexo,
        try_cast(data_nascimento AS DATE) AS data_nascimento,
        INITCAP(TRIM(cidade)) AS cidade,
        UPPER(TRIM(estado)) AS estado,
        try_cast(data_cadastro AS DATE) AS data_cadastro,
        current_timestamp() AS dh_atualizacao
    FROM `desafio-semana-2`.bronze.bronze_clientes
    WHERE id_cliente IS NOT NULL AND TRIM(id_cliente) != ''
) AS src
ON dest.id_cliente = src.id_cliente
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Cria tabela Silver de vendas com schema tipado (vazia)
CREATE TABLE IF NOT EXISTS `desafio-semana-2`.silver.silver_vendas (
    id_venda INT,
    id_cliente INT,
    data_venda DATE,
    produto STRING,
    quantidade INT,
    valor_total DECIMAL(10,2),
    dh_atualizacao TIMESTAMP
) USING DELTA;

In [0]:
%sql
-- MERGE: tipagem (INT, DATE, DECIMAL), padronização (TRIM, INITCAP), filtro de quantidade invalida (<=0)
-- BONUS: idempotente — pode reexecutar sem duplicar dados
MERGE INTO `desafio-semana-2`.silver.silver_vendas AS dest
USING (
    SELECT
        try_cast(id_venda AS INT) AS id_venda,
        try_cast(id_cliente AS INT) AS id_cliente,
        try_cast(data_venda AS DATE) AS data_venda,
        INITCAP(TRIM(produto)) AS produto,
        try_cast(quantidade AS INT) AS quantidade,
        try_cast(valor_total AS DECIMAL(10,2)) AS valor_total,
        current_timestamp() AS dh_atualizacao
    FROM `desafio-semana-2`.bronze.bronze_vendas
    WHERE id_venda IS NOT NULL AND TRIM(id_venda) != ''
      AND try_cast(quantidade AS INT) > 0
) AS src
ON dest.id_venda = src.id_venda
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Cria tabela Silver de suporte com schema tipado (vazia)
CREATE TABLE IF NOT EXISTS `desafio-semana-2`.silver.silver_suporte (
    id_interacao INT,
    id_cliente INT,
    canal STRING,
    tipo_problema STRING,
    tempo_resolucao INT,
    satisfacao_cliente INT,
    dh_atualizacao TIMESTAMP
) USING DELTA;

In [0]:
%sql
-- MERGE: tipagem (INT), padronização (TRIM, INITCAP), filtro de satisfacao invalida (fora de 1-5)
-- BONUS: idempotente — pode reexecutar sem duplicar dados
MERGE INTO `desafio-semana-2`.silver.silver_suporte AS dest
USING (
    SELECT
        try_cast(id_interacao AS INT) AS id_interacao,
        try_cast(id_cliente AS INT) AS id_cliente,
        INITCAP(TRIM(canal)) AS canal,
        INITCAP(TRIM(tipo_problema)) AS tipo_problema,
        try_cast(tempo_resolucao AS INT) AS tempo_resolucao,
        try_cast(satisfacao_cliente AS INT) AS satisfacao_cliente,
        current_timestamp() AS dh_atualizacao
    FROM `desafio-semana-2`.bronze.bronze_suporte
    WHERE id_interacao IS NOT NULL AND TRIM(id_interacao) != ''
      AND try_cast(satisfacao_cliente AS INT) BETWEEN 1 AND 5
) AS src
ON dest.id_interacao = src.id_interacao
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Amostra das 3 tabelas Silver para validar transformacoes
SELECT 'silver_clientes' AS tabela, * FROM `desafio-semana-2`.silver.silver_clientes LIMIT 5;
SELECT 'silver_vendas' AS tabela, * FROM `desafio-semana-2`.silver.silver_vendas LIMIT 5;
SELECT 'silver_suporte' AS tabela, * FROM `desafio-semana-2`.silver.silver_suporte LIMIT 5;

In [0]:
%sql
-- RELATORIO DE INTEGRACAO (item D do desafio — vale 20 pontos)
-- Identifica anomalias entre as 3 fontes

-- 1. Vendas sem cliente cadastrado
SELECT 'Vendas sem cliente' AS anomalia, COUNT(*) AS total
FROM `desafio-semana-2`.silver.silver_vendas v
LEFT JOIN `desafio-semana-2`.silver.silver_clientes c ON v.id_cliente = c.id_cliente
WHERE c.id_cliente IS NULL

UNION ALL

-- 2. Chamados de suporte sem cliente cadastrado
SELECT 'Chamados sem cliente', COUNT(*)
FROM `desafio-semana-2`.silver.silver_suporte s
LEFT JOIN `desafio-semana-2`.silver.silver_clientes c ON s.id_cliente = c.id_cliente
WHERE c.id_cliente IS NULL

UNION ALL

-- 3. Clientes duplicados
SELECT 'Clientes duplicados', COUNT(*)
FROM (
    SELECT id_cliente, COUNT(*) AS qtd
    FROM `desafio-semana-2`.silver.silver_clientes
    GROUP BY id_cliente
    HAVING COUNT(*) > 1
)

UNION ALL

-- 4. Vendas com id_cliente nulo
SELECT 'Vendas com id_cliente nulo', COUNT(*)
FROM `desafio-semana-2`.silver.silver_vendas
WHERE id_cliente IS NULL

UNION ALL

-- 5. Chamados com id_cliente nulo
SELECT 'Chamados com id_cliente nulo', COUNT(*)
FROM `desafio-semana-2`.silver.silver_suporte
WHERE id_cliente IS NULL;

In [0]:
%sql
-- METRICAS DE QUALIDADE DAS TABELAS SILVER
SELECT 'silver_clientes' AS tabela, COUNT(*) AS total_registros
FROM `desafio-semana-2`.silver.silver_clientes
UNION ALL
SELECT 'silver_vendas', COUNT(*)
FROM `desafio-semana-2`.silver.silver_vendas
UNION ALL
SELECT 'silver_suporte', COUNT(*)
FROM `desafio-semana-2`.silver.silver_suporte;